<a href="https://colab.research.google.com/github/MavrinIlua/Mavrin-Ilua-Borisovich_ResumeAndPortfolio/blob/main/%D0%A0%D0%B0%D0%B7%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%BA%D0%B0_%D0%B0%D1%80%D0%BA%D0%B0%D0%B4%D0%BD%D0%BE%D0%B9_%D1%81%D1%82%D1%80%D0%B5%D0%BB%D1%8F%D0%BB%D0%BA%D0%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task
Create a simple 2D game using JavaScript and HTML canvas. The game should include: initializing game variables and player object; implementing player controls for vertical movement; projectile mechanics (shooting); generating and rendering enemies (hearts) from the top of the screen; an main game loop for updating and rendering game elements; collision detection between projectiles and enemies (with scoring) and between enemies and the bottom of the screen (with penalties); a score counter and 'Game Over' logic; a game restart function; and basic visual enhancements.

## Инициализация игровых переменных и объекта игрока

### Subtask:
Определим основные игровые переменные (ширина/высота canvas, состояние игры) и создадим объект игрока с его позицией, размерами, скоростью и методом для отрисовки на canvas.


**Reasoning**:
I need to initialize the game variables and the player object as per the subtask instructions. First, I will create a script block within the existing HTML to get a reference to the canvas element and its 2D rendering context. I will then update the `html_content` variable and display it.



In [3]:
html_content = '''
<!DOCTYPE html>
<html>
<head>
    <title>Game Canvas</title>
    <style>
        body {
            margin: 0;
            overflow: hidden;
            display: flex;
            justify-content: center;
            align-items: center;
            min-height: 100vh;
            background-color: #333;
        }
        #gameCanvas {
            background-color: #000;
            border: 2px solid #fff;
            width: 800px;
            height: 600px;
        }
    </style>
</head>
<body>
    <canvas id="gameCanvas"></canvas>
    <script>
        const canvas = document.getElementById('gameCanvas');
        const ctx = canvas.getContext('2d');

        // Set actual canvas dimensions
        canvas.width = 800;
        canvas.height = 600;

        // Game variables
        const GAME_WIDTH = canvas.width;
        const GAME_HEIGHT = canvas.height;
        let gameState = 'running'; // 'running', 'game over'
        let score = 0;
        let lives = 3; // Initialize lives

        // Player object
        const player = {
            width: 50,
            height: 50,
            x: GAME_WIDTH / 2 - 25,
            y: GAME_HEIGHT - 70,
            speed: 5,
            draw: function() {
                ctx.fillStyle = 'blue';
                ctx.beginPath();
                ctx.moveTo(this.x + this.width / 2, this.y); // Top center
                ctx.lineTo(this.x, this.y + this.height); // Bottom-left
                ctx.lineTo(this.x + this.width, this.y + this.height); // Bottom-right
                ctx.closePath();
                ctx.fill();
            }
        };

        // Projectiles array
        const projectiles = [];

        // Projectile constructor
        function Projectile(x, y) {
            this.width = 10;
            this.height = 20;
            this.x = x;
            this.y = y;
            this.speed = 7;
            this.draw = function() {
                ctx.fillStyle = 'red';
                ctx.beginPath();
                ctx.arc(this.x + this.width / 2, this.y + this.height / 2, this.width / 2, 0, Math.PI * 2);
                ctx.fill();
            };
        }

        // Enemies array
        const enemies = [];

        // Enemy constructor
        function Enemy(x, y) {
            this.width = 40;
            this.height = 40;
            this.x = x;
            this.y = y;
            this.speed = 2; // Slower than player speed
            this.draw = function() {
                // Draw a heart shape
                ctx.fillStyle = 'green'; // Color of the heart
                ctx.beginPath();

                // Approximate heart shape using Bezier curves
                const topX = this.x + this.width / 2;
                const topY = this.y + this.height * 0.2;
                const controlOffset = this.width * 0.4; // Adjust for heart plumpness

                ctx.moveTo(topX, topY);

                // Left curve
                ctx.bezierCurveTo(
                    topX - controlOffset, topY - this.height * 0.2, // Control point 1
                    this.x - this.width * 0.1, this.y + this.height * 0.5, // Control point 2
                    topX, this.y + this.height // Bottom point
                );

                // Right curve
                ctx.bezierCurveTo(
                    this.x + this.width + this.width * 0.1, this.y + this.height * 0.5, // Control point 2
                    topX + controlOffset, topY - this.height * 0.2, // Control point 1
                    topX, topY // Back to top point
                );
                ctx.closePath();
                ctx.fill();

            };
        }

        // Function to generate a new enemy
        function generateEnemy() {
            const enemyWidth = 40;
            const randomX = Math.random() * (GAME_WIDTH - enemyWidth);
            const newEnemy = new Enemy(randomX, -enemyWidth); // Start above canvas
            enemies.push(newEnemy);
        }

        let enemyGenerationIntervalId; // Declare here so it's accessible globally

        // Function to reset the game
        function resetGame() {
            score = 0;
            lives = 3;
            gameState = 'running';
            projectiles.length = 0;
            enemies.length = 0;
            player.x = GAME_WIDTH / 2 - player.width / 2; // Reset player position
            player.y = GAME_HEIGHT - 70;
            enemyGenerationIntervalId = setInterval(generateEnemy, 2000); // Restart enemy generation
        }

        // Function to clear the canvas
        function clearCanvas() {
            ctx.clearRect(0, 0, GAME_WIDTH, GAME_HEIGHT);
        }

        // Player movement handling
        document.addEventListener('keydown', (e) => {
            switch (e.key) {
                case 'w':
                case 'W':
                case 'ArrowUp':
                    player.y = Math.max(0, player.y - player.speed);
                    e.preventDefault();
                    break;
                case 's':
                case 'S':
                case 'ArrowDown':
                    player.y = Math.min(GAME_HEIGHT - player.height, player.y + player.speed);
                    e.preventDefault();
                    break;
                case 'a':
                case 'A':
                case 'ArrowLeft':
                    player.x = Math.max(0, player.x - player.speed);
                    e.preventDefault();
                    break;
                case 'd':
                case 'D':
                case 'ArrowRight':
                    player.x = Math.min(GAME_WIDTH - player.width, player.x + player.speed);
                    e.preventDefault();
                    break;
                case ' ': // Spacebar to shoot
                    const newProjectile = new Projectile(
                        player.x + player.width / 2 - 5, // Center projectile on player
                        player.y // Start at player's y position
                    );
                    projectiles.push(newProjectile);
                    e.preventDefault();
                    break;
                case 'r':
                case 'R':
                    if (gameState === 'game over') {
                        resetGame();
                    }
                    break;
            }
        });

        // Update game elements
        function update() {
            // Update projectiles
            for (let i = projectiles.length - 1; i >= 0; i--) {
                projectiles[i].y -= projectiles[i].speed;
                if (projectiles[i].y + projectiles[i].height < 0) {
                    projectiles.splice(i, 1); // Remove if off-screen
                }
            }

            // Update enemies
            for (let i = enemies.length - 1; i >= 0; i--) {
                enemies[i].y += enemies[i].speed;
                // Check for enemies going off screen at the bottom
                if (enemies[i].y > GAME_HEIGHT) {
                    lives--; // Decrement lives
                    enemies.splice(i, 1); // Remove enemy
                    if (lives <= 0) {
                        gameState = 'game over';
                        clearInterval(enemyGenerationIntervalId); // Stop enemy generation
                    }
                }
            }

            // Collision detection (Projectile - Enemy)
            for (let i = projectiles.length - 1; i >= 0; i--) {
                let projectile = projectiles[i];
                for (let j = enemies.length - 1; j >= 0; j--) {
                    let enemy = enemies[j];

                    // Check for collision using bounding box intersection
                    if (projectile.x < enemy.x + enemy.width &&
                        projectile.x + projectile.width > enemy.x &&
                        projectile.y < enemy.y + enemy.height &&
                        projectile.y + projectile.height > enemy.y) {

                        // Collision detected!
                        score++; // Increment score
                        projectiles.splice(i, 1); // Remove projectile
                        enemies.splice(j, 1); // Remove enemy

                        // Decrement i and j to account for removed elements
                        // If projectile is removed, the current i is invalid. The outer loop will decrement it.
                        // If enemy is removed, the current j is invalid. The inner loop will decrement it.

                        break; // Break from inner loop as this projectile has hit an enemy
                    }
                }
            }
        }

        // Draw all game elements
        function drawAll() {
            player.draw();
            projectiles.forEach(p => p.draw());
            enemies.forEach(e => e.draw());

            // Display score and lives
            ctx.fillStyle = 'white';
            ctx.font = '20px Arial';
            ctx.fillText('Score: ' + score, 10, 25);
            ctx.fillText('Lives: ' + lives, 10, 50);

            if (gameState === 'game over') {
                ctx.fillStyle = 'red';
                ctx.font = '48px Arial';
                ctx.textAlign = 'center';
                ctx.fillText('GAME OVER', GAME_WIDTH / 2, GAME_HEIGHT / 2);
                ctx.font = '24px Arial';
                ctx.fillText('Press R to Restart', GAME_WIDTH / 2, GAME_HEIGHT / 2 + 40);
            }
        }

        // Main game loop
        function gameLoop() {
            if (gameState === 'running') {
                clearCanvas();
                update();
                drawAll();
            } else if (gameState === 'game over') {
                // Only draw game over screen if game is over, don't update anything
                clearCanvas();
                drawAll(); // This will draw the game over text
            }
            requestAnimationFrame(gameLoop);
        }

        // Start the game loop
        gameLoop();

        // Generate enemies periodically
        enemyGenerationIntervalId = setInterval(generateEnemy, 2000); // Initialize the interval ID

        console.log('Canvas and context initialized:', canvas, ctx);
        console.log('Game variables initialized:', { GAME_WIDTH, GAME_HEIGHT, gameState, score, lives });
        console.log('Player object initialized:', player);
        console.log('Projectiles array initialized:', projectiles);
        console.log('Enemies array initialized:', enemies);

    </script>
</body>
</html>
'''

from IPython.display import HTML
HTML(html_content)

## Реализация управления игроком

### Subtask:
Добавим обработчики событий для клавиш 'W'/'S' или 'Вверх'/'Вниз', чтобы управлять вертикальным перемещением игрока, убедившись, что он не выходит за пределы canvas.


## Механика выстрелов

### Subtask:
Реализуем создание объектов снарядов по нажатию левой кнопки мыши или пробела. Определим их свойства (позиция, скорость) и метод для отрисовки, а также логику их движения.


## Генерация и отрисовка врагов (сердечек)

### Subtask:
Создадим объект 'сердечко' с его свойствами (позиция, скорость, размер). Напишем функцию для случайной генерации сердечек сверху экрана и добавим их в массив, а также функцию для их отрисовки.


## Обновление состояния игры и основной игровой цикл

### Subtask:
Создадим основной игровой цикл с помощью `requestAnimationFrame`, который будет очищать canvas, обновлять позиции игрока, снарядов и врагов, а затем перерисовывать все элементы для анимации.


## Обнаружение столкновений (снаряд-враг)

### Subtask:
Напишем функцию для проверки столкновений между снарядами и сердечками. При столкновении удаляем снаряд и сердечко, и начисляем очки.


## Обнаружение столкновений (враг-земля)

### Subtask:
Реализуем проверку столкновений сердечек с нижней границей экрана. При достижении низа экрана применяем штраф (например, уменьшаем 'жизни') и удаляем сердечко.


# Task
Implement a game restart function (`resetGame`) in JavaScript, triggered by the 'R' key when the game state is 'game over'. Enhance visuals by making the player a distinct polygonal shape, projectiles as circles, and enemies as heart shapes. Finally, provide a summary of all implemented game features.

## Исправление движения игрока (влево/вправо)

### Subtask:
Модифицировать объект игрока, чтобы он мог двигаться по горизонтали (`player.x`) в пределах Canvas. Обновить обработчик события `keydown` для клавиш 'A'/'D' или 'ArrowLeft'/'ArrowRight' и добавить `e.preventDefault()` для этих клавиш, чтобы предотвратить прокрутку страницы и 'дёрганье' браузера.
